# Instalar dependencias

In [1]:
import subprocess

print('Instalando dependencias...')
subprocess.run(['pip', 'install', 'numpy', '--upgrade', '-q'])
subprocess.run(['pip', 'install', 'scipy', 'nibabel', 'pandas', 'tqdm', '-q'])
subprocess.run(['pip', 'install', 'nnunetv2', '-q'])
print('✓ Dependencias instaladas')

print('Instalando rclone...')
subprocess.run(['apt-get', 'install', '-y', 'rclone'], capture_output=True)
print('✓ rclone instalado')

Instalando dependencias...
✓ Dependencias instaladas
Instalando rclone...
✓ rclone instalado


#  Montar Drive y configurar rutas

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import os
from pathlib import Path

DRIVE_BASE    = '/content/drive/MyDrive/VerSe_2020_Dataset/preprocessed_verse_for_training'
DRIVE_RESULTS = '/content/drive/MyDrive/VerSe_2020_Dataset/results'
DRIVE_NNUNET = '/content/drive/MyDrive/VerSe_2020_Dataset/MedNeXt_training/mednext/Dataset507_VerSe2020'  # ← mednext

BASE_DIR       = '/content/nnunet_verse'
NNUNET_RAW     = f'{BASE_DIR}/nnUNet_raw'
NNUNET_PREPROC = f'{BASE_DIR}/nnUNet_preprocessed'
NNUNET_RESULTS = f'{BASE_DIR}/nnUNet_results'
PREDICTIONS    = f'{BASE_DIR}/predictions'

for d in [NNUNET_RAW, NNUNET_PREPROC, NNUNET_RESULTS, PREDICTIONS]:
    os.makedirs(d, exist_ok=True)

os.environ['nnUNet_raw']          = NNUNET_RAW
os.environ['nnUNet_preprocessed'] = NNUNET_PREPROC
os.environ['nnUNet_results']      = NNUNET_RESULTS
os.environ['nnUNet_n_proc_DA']    = '16'

DATASET_ID   = 507
DATASET_NAME = f'Dataset{DATASET_ID:03d}_VerSe2020'
TRAINER      = 'nnUNetTrainerMedNeXt_250epochs'  # ← mednext
CONFIG       = '3d_lowres'

print('✓ Rutas configuradas')
print(f'  TRAINER: {TRAINER}')

✓ Rutas configuradas
  TRAINER: nnUNetTrainerMedNeXt_250epochs


# Restaurar preprocessing desde Drive

In [ ]:
import shutil
from pathlib import Path
from tqdm.notebook import tqdm

def copy_with_shutil(src, dst, desc='Copiando'):
    src = Path(src)
    dst = Path(dst)
    dst.mkdir(parents=True, exist_ok=True)

    all_files   = [f for f in src.rglob('*') if f.is_file()]
    total_files = len(all_files)
    total_gb    = round(sum(f.stat().st_size for f in all_files) / 1e9, 2)
    print(f'{desc}: {total_files} archivos ({total_gb} GB)')

    for f in tqdm(all_files, desc=desc,
                  bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]'):
        dst_f = dst / f.relative_to(src)
        dst_f.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(f), str(dst_f))

    print(f'✓ {desc} completado\n')

print('Restaurando preprocessing desde Drive...\n')

copy_with_shutil(
    Path(DRIVE_BASE) / 'nnUNet_preprocessed' / DATASET_NAME / 'nnUNetPlans_3d_lowres',
    Path(NNUNET_PREPROC) / DATASET_NAME / 'nnUNetPlans_3d_lowres',
    'nnUNetPlans_3d_lowres'
)

copy_with_shutil(
    Path(DRIVE_BASE) / 'nnUNet_preprocessed' / DATASET_NAME / 'gt_segmentations',
    Path(NNUNET_PREPROC) / DATASET_NAME / 'gt_segmentations',
    'gt_segmentations'
)

print('Copiando JSONs...')
for json_file in ['nnUNetPlans.json', 'dataset.json', 'dataset_fingerprint.json', 'splits_final.json']:
    src = Path(DRIVE_BASE) / 'nnUNet_preprocessed' / DATASET_NAME / json_file
    dst = Path(NNUNET_PREPROC) / DATASET_NAME / json_file
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.exists():
        shutil.copy2(str(src), str(dst))
        print(f'  ✓ {json_file}')

for json_file in ['dataset.json', 'splits_info.json']:
    src = Path(DRIVE_BASE) / 'nnUNet_raw' / DATASET_NAME / json_file
    dst = Path(NNUNET_RAW) / DATASET_NAME / json_file
    Path(dst).parent.mkdir(parents=True, exist_ok=True)
    if src.exists():
        shutil.copy2(str(src), str(dst))
        print(f'  ✓ {json_file}')

# imagesTs y labelsTs
copy_with_shutil(
    Path(DRIVE_BASE) / 'nnUNet_raw' / DATASET_NAME / 'imagesTs',
    Path(NNUNET_RAW) / DATASET_NAME / 'imagesTs',
    'imagesTs'
)

copy_with_shutil(
    Path(DRIVE_BASE) / 'nnUNet_raw' / DATASET_NAME / 'labelsTs',
    Path(NNUNET_RAW) / DATASET_NAME / 'labelsTs',
    'labelsTs'
)

n_lowres = len(list((Path(NNUNET_PREPROC) / DATASET_NAME / 'nnUNetPlans_3d_lowres').glob('*')))
n_img    = len(list((Path(NNUNET_RAW) / DATASET_NAME / 'imagesTs').glob('*.nii.gz')))
n_lbl    = len(list((Path(NNUNET_RAW) / DATASET_NAME / 'labelsTs').glob('*.nii.gz')))

print(f'\n✓ Verificación final:')
print(f'  nnUNetPlans_3d_lowres: {n_lowres} archivos')
print(f'  imagesTs:              {n_img} CTs')
print(f'  labelsTs:              {n_lbl} máscaras')

Restaurando preprocessing desde Drive...

nnUNetPlans_3d_lowres: 783 archivos (4.42 GB)


nnUNetPlans_3d_lowres:   0%|          | 0/783 [00:00<?]

✓ nnUNetPlans_3d_lowres completado

gt_segmentations: 261 archivos (0.15 GB)


gt_segmentations:   0%|          | 0/261 [00:00<?]

✓ gt_segmentations completado

Copiando JSONs...
  ✓ nnUNetPlans.json
  ✓ dataset.json
  ✓ dataset_fingerprint.json
  ✓ splits_final.json
  ✓ dataset.json
  ✓ splits_info.json
imagesTs: 113 archivos (18.04 GB)


imagesTs:   0%|          | 0/113 [00:00<?]

✓ imagesTs completado

labelsTs: 113 archivos (0.08 GB)


labelsTs:   0%|          | 0/113 [00:00<?]

✓ labelsTs completado


✓ Verificación final:
  nnUNetPlans_3d_lowres: 783 archivos
  imagesTs:              113 CTs
  labelsTs:              113 máscaras


#  Restaurar checkpoints de nnU-Net desde Drive

In [ ]:
import shutil
from pathlib import Path

trainer_dst = Path(NNUNET_RESULTS) / DATASET_NAME / \
              f'nnUNetTrainerMedNeXt_250epochs__nnUNetPlans__{CONFIG}'
trainer_dst.mkdir(parents=True, exist_ok=True)

drive_trainer = Path(DRIVE_NNUNET) / f'nnUNetTrainerMedNeXt_250epochs__nnUNetPlans__{CONFIG}'

for json_file in ['dataset.json', 'dataset_fingerprint.json', 'plans.json']:
    for src_dir in [drive_trainer, Path(DRIVE_NNUNET)]:
        src = src_dir / json_file
        if src.exists():
            shutil.copy2(str(src), str(trainer_dst / json_file))
            print(f'✓ {json_file}')
            break
    else:
        src = Path(NNUNET_RESULTS) / DATASET_NAME / json_file
        if src.exists():
            shutil.copy2(str(src), str(trainer_dst / json_file))
            print(f'✓ {json_file} (desde local)')
        else:
            print(f'✗ {json_file} no encontrado')

✓ dataset.json
✓ dataset_fingerprint.json
✓ plans.json


In [ ]:
import shutil
from pathlib import Path

print('Restaurando checkpoints de MedNeXt...')

trainer_folder = f'nnUNetTrainerMedNeXt_250epochs__nnUNetPlans__{CONFIG}'
src_base = Path(DRIVE_NNUNET) / trainer_folder
dst_base = Path(NNUNET_RESULTS) / DATASET_NAME / trainer_folder
dst_base.mkdir(parents=True, exist_ok=True)

for fold in range(5):
    src_fold = src_base / f'fold_{fold}'
    dst_fold = dst_base / f'fold_{fold}'

    if src_fold.exists():
        dst_fold.mkdir(parents=True, exist_ok=True)
        shutil.copytree(str(src_fold), str(dst_fold), dirs_exist_ok=True)
        has_final = (dst_fold / 'checkpoint_final.pth').exists()
        has_best  = (dst_fold / 'checkpoint_best.pth').exists()
        print(f'  ✓ Fold {fold} — final: {has_final}, best: {has_best}')
    else:
        print(f'  ✗ Fold {fold} — no encontrado en: {src_fold}')

for json_file in ['plans.json', 'dataset.json', 'dataset_fingerprint.json']:
    for search_path in [src_base, src_base.parent]:
        src_j = search_path / json_file
        if src_j.exists():
            shutil.copy(str(src_j), str(dst_base.parent / json_file))
            print(f'  ✓ {json_file}')
            break

print('\n✓ Checkpoints restaurados')

Restaurando checkpoints de MedNeXt...
  ✓ Fold 0 — final: True, best: True
  ✓ Fold 1 — final: True, best: True
  ✓ Fold 2 — final: True, best: True
  ✓ Fold 3 — final: True, best: True
  ✓ Fold 4 — final: True, best: True
  ✓ plans.json
  ✓ dataset.json
  ✓ dataset_fingerprint.json

✓ Checkpoints restaurados


# Instalar trainer 250 epochs

In [ ]:
import nnunetv2
from pathlib import Path

nnunet_dir  = Path(nnunetv2.__file__).parent
trainer_dir = nnunet_dir / 'training' / 'nnUNetTrainer' / 'variants' / 'network_architecture'
trainer_dir.mkdir(parents=True, exist_ok=True)
trainer_code = '''
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
from nnunet_mednext import create_mednext_v1
import torch
import warnings

class MedNeXtWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model   = model
        self.encoder = model
        self.decoder = model

    def forward(self, x):
        output = self.model(x)
        if self.training:
            # Durante entrenamiento devuelve lista para deep supervision
            if isinstance(output, (list, tuple)):
                return output
            return [output]
        else:
            # Durante inferencia devuelve solo el tensor principal
            if isinstance(output, (list, tuple)):
                return output[0]
            return output

    def load_state_dict(self, state_dict, strict=True):
        model_sd = {}
        for k, v in state_dict.items():
            if k.startswith('model.'):
                model_sd[k[len('model.'):]] = v
        return self.model.load_state_dict(model_sd, strict=False)

class nnUNetTrainerMedNeXt_250epochs(nnUNetTrainer):
    max_num_epochs = 250
    compile = False

    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.initial_lr = 1e-3
        self.num_epochs = 250
        warnings.filterwarnings("ignore", message=".*lr_scheduler.step.*")
        warnings.filterwarnings("ignore", message=".*epoch parameter.*")

    @staticmethod
    def build_network_architecture(plans_manager, configuration_manager,
                                   num_input_channels, num_output_channels,
                                   enable_deep_supervision=True):
        base_model = create_mednext_v1(
            num_input_channels=num_input_channels,
            num_classes=num_output_channels,
            model_id="B",
            kernel_size=3,
            deep_supervision=enable_deep_supervision
        )
        return MedNeXtWrapper(base_model)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.network.parameters(),
            lr=self.initial_lr,
            weight_decay=3e-5
        )
        lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=self.num_epochs,
            eta_min=1e-6
        )
        return optimizer, lr_scheduler

    def set_deep_supervision_enabled(self, enabled: bool):
        pass

    def perform_actual_validation(self, save_probabilities: bool = False):
        self.logger.log("Mean Foreground Dice", 0, self.current_epoch - 1)
        return

    def on_train_end(self):
        self.inference_allowed_mirroring_axes = None
        super().on_train_end()
'''

# Instalar MedNeXt
import subprocess
subprocess.run(['pip', 'install', 'git+https://github.com/MIC-DKFZ/MedNeXt.git', '-q'],
               capture_output=True)

trainer_path = trainer_dir / 'nnUNetTrainerMedNeXt_250epochs.py'
trainer_path.write_text(trainer_code)

result = subprocess.run(
    ['python', '-c',
     'from nnunetv2.training.nnUNetTrainer.variants.network_architecture.nnUNetTrainerMedNeXt_250epochs import nnUNetTrainerMedNeXt_250epochs; print("✓ trainer MedNeXt instalado")'],
    capture_output=True, text=True
)
print(result.stdout if result.returncode == 0 else f'✗ {result.stderr}')

✓ trainer MedNeXt instalado



# Inferencia con ensemble de 5 folds

In [ ]:
import os
import subprocess
import threading
import time
import shutil
from pathlib import Path

input_dir  = f'{NNUNET_RAW}/{DATASET_NAME}/imagesTs'
output_dir = f'{NNUNET_RESULTS}/{DATASET_NAME}/inference_mednext_3d_lowres'
os.makedirs(output_dir, exist_ok=True)

total_casos = len(list(Path(input_dir).glob('*.nii.gz')))

stop_flag = [False]
def monitor_inferencia():
    while not stop_flag[0]:
        n_pred = len(list(Path(output_dir).glob('*.nii.gz')))
        pct    = int(n_pred / total_casos * 100) if total_casos > 0 else 0
        bar    = '█' * (pct // 5) + '░' * (20 - pct // 5)
        print(f'\r  [{bar}] {n_pred}/{total_casos} casos ({pct}%) — {time.strftime("%H:%M:%S")}', end='')
        time.sleep(10)

print(f'=== Inferencia MedNeXt 3d_lowres ===')
print(f'  Input:    {input_dir}')
print(f'  Output:   {output_dir}')
print(f'  Casos:    {total_casos}')
print(f'  Ensemble: 5 folds\n')

t = threading.Thread(target=monitor_inferencia, daemon=True)
t.start()

cmd = [
    'nnUNetv2_predict',
    '-i', input_dir,
    '-o', output_dir,
    '-d', str(DATASET_ID),
    '-c', CONFIG,
    '-tr', TRAINER,
    '-f', '0', '1', '2', '3', '4',
    '-step_size', '0.5',
]
result = subprocess.run(cmd, text=True)

stop_flag[0] = True
t.join(timeout=5)

if result.returncode == 0:
    n_pred = len(list(Path(output_dir).glob('*.nii.gz')))
    print(f'\n✓ Inferencia completada — {n_pred} predicciones')
    drive_inf = Path(DRIVE_RESULTS) / 'mednext' / 'inference_3d_lowres'
    shutil.copytree(output_dir, str(drive_inf), dirs_exist_ok=True)
    print(f'✓ Predicciones → Drive: {drive_inf}')
else:
    print(f'\n✗ Error en inferencia: {result.returncode}')

=== Inferencia MedNeXt 3d_lowres ===
  Input:    /content/nnunet_verse/nnUNet_raw/Dataset507_VerSe2020/imagesTs
  Output:   /content/nnunet_verse/nnUNet_results/Dataset507_VerSe2020/inference_mednext_3d_lowres
  Casos:    113
  Ensemble: 5 folds

  [███████████████████░] 112/113 casos (99%) — 08:20:01
✓ Inferencia completada — 113 predicciones
✓ Predicciones → Drive: /content/drive/MyDrive/VerSe_2020_Dataset/results/mednext/inference_3d_lowres


# Post Procesado

In [ ]:
import numpy as np
import nibabel as nib
from scipy import ndimage
from pathlib import Path
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', message='.*copy keyword.*')

pred_dir = Path('/content/drive/MyDrive/VerSe_2020_Dataset/results/mednext/inference_3d_lowres')
post_dir = Path('/content/drive/MyDrive/VerSe_2020_Dataset/results/mednext/inference_3d_lowres_postprocessed')
post_dir.mkdir(parents=True, exist_ok=True)

VERSE_LABEL_TO_NAME = {
    1:'C1',  2:'C2',  3:'C3',  4:'C4',  5:'C5',  6:'C6',  7:'C7',
    8:'T1',  9:'T2',  10:'T3', 11:'T4', 12:'T5', 13:'T6',
    14:'T7', 15:'T8', 16:'T9', 17:'T10',18:'T11',19:'T12',
    20:'L1', 21:'L2', 22:'L3', 23:'L4', 24:'L5', 25:'L6', 26:'S1',
    28:'T13',
}
ALL_LABELS = sorted(VERSE_LABEL_TO_NAME.keys())
LABEL_MAP  = {verse_l: cont_l for cont_l, verse_l in enumerate(ALL_LABELS, start=1)}
all_labels = list(LABEL_MAP.values())

def postprocess(pred_array, labels):
    postprocessed = np.zeros_like(pred_array)
    for label in labels:
        binary = (pred_array == label)
        if not np.any(binary):
            continue
        labeled, n = ndimage.label(binary)
        if n <= 1:
            postprocessed[binary] = label
            continue
        sizes   = ndimage.sum(binary, labeled, range(1, n + 1))
        largest = np.argmax(sizes) + 1
        postprocessed[labeled == largest] = label
    return postprocessed

pred_files = sorted(pred_dir.glob('*.nii.gz'))
print(f'Post-procesando MedNeXt ({len(pred_files)} casos)...')

for pred_path in tqdm(pred_files, desc='Post-procesado',
                      bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]'):
    pred_nib  = nib.load(str(pred_path))
    pred_data = np.asarray(pred_nib.dataobj).astype(np.uint8)
    cleaned   = postprocess(pred_data, all_labels)
    out_nib   = nib.Nifti1Image(cleaned, pred_nib.affine, pred_nib.header)
    out_nib.header.set_data_dtype(np.uint8)
    nib.save(out_nib, str(post_dir / pred_path.name))

n = len(list(post_dir.glob('*.nii.gz')))
print(f'✓ Post-procesado completado — {n} casos')
PRED_FINAL_MEDNEXT = str(post_dir)

Post-procesando MedNeXt (113 casos)...


Post-procesado:   0%|          | 0/113 [00:00<?]

✓ Post-procesado completado — 113 casos


# Evaluacion DSC + ID Rate + RMSD

In [6]:
import shutil
from pathlib import Path

src = Path(DRIVE_BASE) / 'nnUNet_raw' / DATASET_NAME / 'labelsTs'
dst = Path(NNUNET_RAW) / DATASET_NAME / 'labelsTs'

if not dst.exists() or len(list(dst.glob('*.nii.gz'))) == 0:
    print('Copiando labelsTs...')
    shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
    print(f'✓ labelsTs: {len(list(dst.glob("*.nii.gz")))} casos')
else:
    print(f'✓ labelsTs ya existe: {len(list(dst.glob("*.nii.gz")))} casos')

Copiando labelsTs...
✓ labelsTs: 113 casos


In [7]:
PRED_FINAL_MEDNEXT = '/content/drive/MyDrive/VerSe_2020_Dataset/results/mednext/inference_3d_lowres_postprocessed'
GT_DIR_LOCAL = f'{NNUNET_RAW}/{DATASET_NAME}/labelsTs'

# Verificar que existen
from pathlib import Path
print(f'Predicciones: {len(list(Path(PRED_FINAL_MEDNEXT).glob("*.nii.gz")))} casos')
print(f'Ground truth: {len(list(Path(GT_DIR_LOCAL).glob("*.nii.gz")))} casos')

Predicciones: 113 casos
Ground truth: 113 casos


In [8]:
import numpy as np
import nibabel as nib
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import subprocess
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', message='.*copy keyword.*')

subprocess.run(['pip', 'install', 'pycpd', '-q'], capture_output=True)

pred_dir = Path(PRED_FINAL_MEDNEXT)
gt_dir   = Path(f'{NNUNET_RAW}/{DATASET_NAME}/labelsTs')

LABEL_MAP = {
    'background': 0, 'C1': 1, 'C2': 2, 'C3': 3, 'C4': 4, 'C5': 5,
    'C6': 6, 'C7': 7, 'T1': 8, 'T2': 9, 'T3': 10, 'T4': 11, 'T5': 12,
    'T6': 13, 'T7': 14, 'T8': 15, 'T9': 16, 'T10': 17, 'T11': 18,
    'T12': 19, 'L1': 20, 'L2': 21, 'L3': 22, 'L4': 23, 'L5': 24,
    'L6': 25, 'S1': 26, 'T13': 27
}
LABELS = [v for v in LABEL_MAP.values() if v > 0]

def dsc_label(pred, gt, label):
    p, g  = (pred == label), (gt == label)
    inter = np.logical_and(p, g).sum()
    denom = p.sum() + g.sum()
    return float(2 * inter / denom) if denom > 0 else 1.0

def mean_dsc(pred, gt, labels):
    vals = [dsc_label(pred, gt, l) for l in labels if (gt == l).sum() > 0]
    return float(np.mean(vals)) if vals else 0.0

def id_rate(pred, gt, labels, iou_thr=0.5):
    total, correct = 0, 0
    for l in labels:
        if (gt == l).sum() == 0:
            continue
        total += 1
        inter = np.logical_and(pred == l, gt == l).sum()
        union = np.logical_or(pred == l, gt == l).sum()
        if union > 0 and inter / union >= iou_thr:
            correct += 1
    return float(correct / total) if total > 0 else 0.0

def get_centroids(seg, labels, affine):
    cents = {}
    for l in labels:
        idx = np.array(np.where(seg == l)).T
        if len(idx) == 0:
            continue
        c_vox  = idx.mean(axis=0)
        c_phys = (affine @ np.array([*c_vox, 1.0]))[:3]
        cents[l] = c_phys
    return cents

def get_centroids_vox(seg, labels):
    cents = {}
    for l in labels:
        idx = np.array(np.where(seg == l)).T
        if len(idx) == 0:
            continue
        cents[l] = idx.mean(axis=0)
    return cents

def compute_rmsd_vox_mm(pred, gt, affine, labels):
    pc_vox = get_centroids_vox(pred, labels)
    gc_vox = get_centroids_vox(gt,   labels)
    pc_mm  = get_centroids(pred, labels, affine)
    gc_mm  = get_centroids(gt,   labels, affine)

    common = sorted(set(pc_vox) & set(gc_vox))
    if len(common) < 2:
        return float('nan'), float('nan')

    pred_vox = np.array([pc_vox[l] for l in common])
    gt_vox   = np.array([gc_vox[l] for l in common])
    pred_mm  = np.array([pc_mm[l]  for l in common])
    gt_mm    = np.array([gc_mm[l]  for l in common])

    try:
        from pycpd import AffineRegistration
        pred_aligned_mm,  _ = AffineRegistration(X=gt_mm,  Y=pred_mm).register()
        pred_aligned_vox, _ = AffineRegistration(X=gt_vox, Y=pred_vox).register()
    except:
        pred_aligned_mm  = pred_mm  - pred_mm.mean(0)  + gt_mm.mean(0)
        pred_aligned_vox = pred_vox - pred_vox.mean(0) + gt_vox.mean(0)

    rmsd_mm  = float(np.sqrt(np.mean(np.sum((pred_aligned_mm  - gt_mm)  ** 2, axis=1))))
    rmsd_vox = float(np.sqrt(np.mean(np.sum((pred_aligned_vox - gt_vox) ** 2, axis=1))))
    return rmsd_vox, rmsd_mm

pairs = [
    (p, gt_dir / p.name)
    for p in sorted(pred_dir.glob('*.nii.gz'))
    if (gt_dir / p.name).exists()
]

print(f'Evaluando MedNeXt — {len(pairs)} casos...\n')
results = []

for pred_path, gt_path in tqdm(pairs, desc='Evaluando',
                                bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]'):
    try:
        pred_nib = nib.load(str(pred_path))
        gt_nib   = nib.load(str(gt_path))
        pred_arr = np.asarray(pred_nib.dataobj).astype(np.uint8)
        gt_arr   = np.asarray(gt_nib.dataobj).astype(np.uint8)
        rmsd_vox, rmsd_mm = compute_rmsd_vox_mm(pred_arr, gt_arr, gt_nib.affine, LABELS)
        results.append({
            'Case':     pred_path.stem.replace('.nii', ''),
            'DSC':      round(mean_dsc(pred_arr, gt_arr, LABELS),  4),
            'ID_Rate':  round(id_rate(pred_arr, gt_arr, LABELS),   4),
            'RMSD_vox': round(rmsd_vox, 3),
            'RMSD_mm':  round(rmsd_mm,  2),
        })
    except Exception as e:
        print(f'{pred_path.name}: {e}')
        results.append({
            'Case': pred_path.stem.replace('.nii', ''),
            'DSC': float('nan'), 'ID_Rate': float('nan'),
            'RMSD_vox': float('nan'), 'RMSD_mm': float('nan'),
        })

df = pd.DataFrame(results)
mean_row = {
    'Case':     'MEAN',
    'DSC':      round(df['DSC'].mean(), 4),
    'ID_Rate':  round(df['ID_Rate'].mean(), 4),
    'RMSD_vox': round(df['RMSD_vox'].dropna().mean(), 3),
    'RMSD_mm':  round(df['RMSD_mm'].dropna().mean(),  2),
}
df = pd.concat([df, pd.DataFrame([mean_row])], ignore_index=True)

print('\n=== RESULTADOS MedNeXt 3D lowres ===\n')
print(df.to_string(index=False))

mean = df[df['Case'] == 'MEAN'].iloc[0]
print(f'\n{"="*40}')
print(f'RESUMEN FINAL MedNeXt')
print(f'{"="*40}')
print(f'  DSC:     {mean["DSC"]}')
print(f'  ID Rate: {mean["ID_Rate"]}')
print(f'  RMSD:    {mean["RMSD_vox"]} vox  |  {mean["RMSD_mm"]} mm')
print(f'  Casos:   {len(df) - 1}')

output_csv = Path('/content/drive/MyDrive/VerSe_2020_Dataset/results/mednext/mednext_evaluation_3d_lowres.csv')
output_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(str(output_csv), index=False)
print(f'\n✓ Resultados guardados en Drive: {output_csv}')

Evaluando MedNeXt — 113 casos...



Evaluando:   0%|          | 0/113 [00:00<?]


=== RESULTADOS MedNeXt 3D lowres ===

           Case    DSC  ID_Rate  RMSD_vox  RMSD_mm
VerSe_test_0001 0.8680   1.0000     1.929     0.79
VerSe_test_0002 0.8165   0.9167     3.378     1.86
VerSe_test_0003 0.8991   1.0000     1.348     0.44
VerSe_test_0004 0.7871   0.8889     8.609     2.14
VerSe_test_0005 0.8877   1.0000     0.837     0.28
VerSe_test_0006 0.0861   0.0588    25.054    16.98
VerSe_test_0007 0.8871   1.0000     2.410     0.67
VerSe_test_0008 0.8128   1.0000     3.398     1.22
VerSe_test_0009 0.8329   1.0000     1.950     0.74
VerSe_test_0010 0.8466   1.0000     1.668     0.51
VerSe_test_0011 0.8189   0.8824     4.893     3.24
VerSe_test_0012 0.8627   0.8947    10.316    13.14
VerSe_test_0013 0.9386   1.0000     0.503     0.47
VerSe_test_0014 0.7669   0.7647     3.855     3.01
VerSe_test_0015 0.9409   1.0000     0.410     0.31
VerSe_test_0016 0.9310   1.0000     0.317     0.23
VerSe_test_0017 0.8928   0.9444     0.282     0.27
VerSe_test_0018 0.9431   1.0000     0.201  